# 多重繼承與 Mixin（進階選讀）

> **進階選讀：** 本篇為選讀內容。初學者可以先跳過，等遇到實際需求時再回來閱讀。
> 日常開發中，單一繼承加組合已經能解決大部分問題。

## 學習目標

- 理解多重繼承與菱形問題
- 學會查看 MRO（方法解析順序）
- 掌握 Mixin 設計模式
- 知道何時該避免多重繼承

---

## 1. 什麼是多重繼承？

Python 允許一個類別同時繼承多個父類別：

In [ ]:
class A:
    def hello(self):
        return "我是 A"

class B:
    def greet(self):
        return "我是 B"

class C(A, B):      # 同時繼承 A 和 B
    pass

c = C()
print(c.hello())   # 我是 A （從 A 繼承）
print(c.greet())   # 我是 B （從 B 繼承）

## 2. 菱形問題（Diamond Problem）

當繼承關係形成菱形時，問題就來了：

```
      Base
      /  \
     A    B
      \  /
       C
```

In [ ]:
class Base:
    def method(self):
        return "Base"

class A(Base):
    def method(self):
        return "A"

class B(Base):
    def method(self):
        return "B"

class C(A, B):
    pass

print(C().method())  # "A" — 但為什麼不是 "B" 或 "Base"？

答案取決於 **MRO**（Method Resolution Order）。

## 3. MRO — 方法解析順序

Python 使用 C3 線性化演算法決定方法的搜尋順序。你可以用 `__mro__` 查看：

In [ ]:
print(C.__mro__)
# (<class 'C'>, <class 'A'>, <class 'B'>, <class 'Base'>, <class 'object'>)

搜尋順序：`C` → `A` → `B` → `Base` → `object`

所以 `C().method()` 會先在 `C` 找，找不到就去 `A`，在 `A` 找到了就回傳 `"A"`。

**記住規則：** 繼承列表中**越左邊的優先權越高**。`class C(A, B)` 表示 A 優先於 B。

## 4. Mixin 模式 — 多重繼承的正確用法

Mixin 是一種小型、專注單一功能的類別，用來「混入」額外能力。
它通常**不能獨立使用**，只是用來擴充其他類別的功能。

### 範例：LoggerMixin

In [ ]:
import logging

class LoggerMixin:
    """混入日誌功能"""
    def log(self, message, level="info"):
        logger = logging.getLogger(self.__class__.__name__)
        getattr(logger, level)(message)

class UserService(LoggerMixin):
    def create_user(self, name):
        self.log(f"建立使用者: {name}")
        return {"name": name}

service = UserService()
service.create_user("Alice")  # 自動有日誌功能

### 範例：SerializerMixin

In [ ]:
import json

class SerializerMixin:
    """混入序列化功能"""
    def to_dict(self):
        return {
            key: value
            for key, value in self.__dict__.items()
            if not key.startswith("_")
        }

    def to_json(self):
        return json.dumps(self.to_dict(), ensure_ascii=False)

### 組合多個 Mixin

In [ ]:
class Product(SerializerMixin, LoggerMixin):
    def __init__(self, name, price):
        self.name = name
        self.price = price

    def apply_discount(self, percent):
        self.log(f"套用 {percent}% 折扣到 {self.name}")
        new_price = self.price * (1 - percent / 100)
        return Product(self.name, new_price)

p = Product("筆電", 30000)
print(p.to_json())         # {"name": "筆電", "price": 30000}
discounted = p.apply_discount(10)
print(discounted.to_dict())  # {'name': '筆電', 'price': 27000.0}

## 5. Mixin 的命名慣例與設計原則

| 原則 | 說明 |
|---|---|
| 名稱加 `Mixin` 後綴 | 如 `LoggerMixin`，讓人一看就知道用途 |
| 不定義 `__init__` | Mixin 不應該有自己的初始化邏輯 |
| 只提供方法，不存狀態 | 保持簡單，避免屬性衝突 |
| 功能單一 | 一個 Mixin 只做一件事 |
| 放在繼承列表左邊 | `class MyClass(MixinA, MixinB, BaseClass)` |

## 6. 什麼時候該避免多重繼承？

**避免的情況：**
- 兩個父類別有**同名屬性**或**同名方法**（容易衝突）
- 繼承層級超過 3 層（太複雜，難以維護）
- 只是為了「借用」某個方法（用組合更好）

**適合的情況：**
- 加入橫切關注點（日誌、序列化、快取）
- 父類別之間功能完全不重疊
- 使用 Mixin 模式，保持每個類別小而專注

In [ ]:
# 好的用法：Mixin 提供互不重疊的功能
# class AdminUser(AuthMixin, LoggerMixin, User):
#     pass

# 不好的用法：兩個「正式」父類別，容易衝突
# class FlyingCar(Car, Airplane):  # 兩者都有 speed、fuel 等屬性
#     pass

## 7. 練習題

1. 建立 `TimestampMixin`，自動在物件建立時記錄 `created_at` 時間
2. 建立 `ValidatorMixin`，提供 `validate()` 方法檢查所有屬性不為 None
3. 將這兩個 Mixin 混入一個 `Order` 類別中

In [ ]:
# 在這裡寫你的練習程式碼


---

## 重點整理

| 概念 | 說明 |
|---|---|
| 多重繼承 | 一個類別同時繼承多個父類別 |
| 菱形問題 | 多個父類別有共同祖先時的方法衝突 |
| MRO | Python 用 C3 演算法決定方法搜尋順序 |
| `__mro__` | 查看類別的方法解析順序 |
| Mixin | 小型、單一功能的類別，用來混入額外能力 |
| 最佳實踐 | 優先使用 Mixin，避免深層多重繼承 |

---

> 上一篇：[03_special_methods.ipynb](03_special_methods.ipynb) -- 魔術方法